## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [14]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [15]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
ollama_url = "http://localhost:11434/v1"
# MODEL = "gpt-4.1-nano"
MODEL = 'llama3.1:latest'
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [16]:

# Pick an embedding model
from langchain_community.embeddings import OllamaEmbeddings

# Initialize Ollama embeddings
embeddings = OllamaEmbeddings(model=MODEL)
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [17]:
retriever = vectorstore.as_retriever()
# openai = OpenAI()
llm = ChatOpenAI(api_key='ollama', base_url=ollama_url, model_name=MODEL, temperature=0)

### These LangChain objects implement the method `invoke()`

In [ ]:
# Retriever gets the right documents
retriever.invoke("Who is Avery?")

[Document(id='3e45d8e2-d49b-4fa3-8124-eea196f70979', metadata={'source': 'knowledge-base/employees/Emily Carter.md', 'doc_type': 'employees'}, page_content="Emily Carter exemplifies the kind of talent that drives Insurellm's success and is an invaluable asset to the company."),
 Document(id='de328f8c-2cae-4de9-b0b9-21f8efc37428', metadata={'doc_type': 'products', 'source': 'knowledge-base/products/Claimllm.md'}, page_content='Claimllm represents the future of insurance claims—faster, smarter, and more customer-centric. Transform your claims operation and deliver the service your policyholders deserve!'),
 Document(id='84135fbf-689c-41f8-b011-ac9fc27f2628', metadata={'doc_type': 'products', 'source': 'knowledge-base/products/Healthllm.md'}, page_content="Healthllm represents Insurellm's commitment to transforming health insurance through technology that improves outcomes for insurers, providers, and members alike. Join us in building the future of health insurance!"),
 Document(id='ca12

In [19]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery can refer to several things, so I\'ll provide a few possible answers:\n\n1. **Given name**: Avery is a popular given name for both males and females in the United States. It\'s often associated with qualities like strength, courage, and independence.\n2. **Surname**: Avery is also a common surname of English origin, which means "elf counsel" or "wise counselor."\n3. **Fictional characters**:\n\t* Avery Frost (also known as Avery Jessup) is a character from the TV show "Parenthood."\n\t* Avery Bradley is a fictional character in the TV series "Gossip Girl."\n4. **Real people**:\n\t* Avery Bradley is an American professional basketball player who plays for the Los Angeles Clippers.\n\t* Avery Singer is an American artist known for her abstract paintings and sculptures.\n5. **Other meanings**: In some contexts, Avery can also refer to a type of paper or cardstock used in crafting and art projects.\n\nIf you\'re thinking of a specific Avery, could you provide more 

## Time to put this together!

In [20]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    # Adding the context here
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [22]:
answer_question("Who is Averi Lancaster?", [])

"I'm not aware of any information about an individual named Averi Lancaster being associated with Insurellm. I can tell you more about Emily Carter, who is mentioned as a valuable asset to the company, but I don't have any information on Averi Lancaster. Would you like to know more about Emily Carter or one of our other services such as Claimllm, Homellm, or Healthllm?"

## What could possibly come next? 😂

In [23]:
gr.ChatInterface(answer_question).launch()

/Users/sagnikrana/Documents/GitHub/Udemy-LLM-Engineering-Master-AI-Large-Language-Models-Agents/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!